In [ ]:
%run "..\scripts\load_daily_variables.py"

Loading Variables: 100%|██████████| 26/26 [04:30<00:00, 10.42s/it]  


In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from market_data.add_technicals import atrs_from

# Study configuration
ATR_BASE = "5DMA"
ATR_DISTANCE_COLUMN = f"ATRs_from_{ATR_BASE}"
RELATIVE_ATR_COLUMN = "Relative_ATR"
REGIME_COLUMN = "regime_crossing_ma_short_2050200"
OPEN_COLUMN = "Open"
CLOSE_COLUMN = "Close"

RELATIVE_ATR_THRESHOLD = 5
# Can establish thresholds based on the descriptive statistics of the ATRs.
LOWER_THRESHOLD = 1.2
UPPER_THRESHOLD = 2
N = 11

OUTCOMES = ("green", "red", "unchanged")
REGIME_LABELS = {1: "fanned", -1: "crossed"}

trade_direction = 'short'


In [55]:
for symbol_data in tqdm(
    symbols.values(),
    total=len(symbols),
    desc=f"Calculating {ATR_DISTANCE_COLUMN}",
    unit="symbol",
):
    atrs_from(symbol_data.df, ATR_BASE)

Calculating ATRs_from_5DMA: 100%|██████████| 3022/3022 [00:07<00:00, 392.67symbol/s]


Descriptive Statistics

In [56]:
# Pool the configured ATR-distance values from every symbol.
atr_series = []
for symbol_data in tqdm(
    symbols.values(),
    total=len(symbols),
    desc=f"Collecting {ATR_DISTANCE_COLUMN}",
    unit="symbol",
):
    if ATR_DISTANCE_COLUMN in symbol_data.df.columns:
        atr_series.append(
            pd.to_numeric(
                symbol_data.df[ATR_DISTANCE_COLUMN],
                errors="coerce",
            )
        )

atr_distance_values = pd.concat(atr_series, ignore_index=True).rename(
    ATR_DISTANCE_COLUMN
)

# Exclude NaN, missing, and positive/negative infinity from every calculation.
valid_atr_distances = atr_distance_values[np.isfinite(atr_distance_values)]

atr_distance_stats = valid_atr_distances.describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("value")
atr_distance_stats.loc["skew"] = valid_atr_distances.skew()
atr_distance_stats.loc["kurtosis"] = valid_atr_distances.kurtosis()

atr_distance_stats

,value
count,1.108518e+07
mean,3.041530e-02
std,7.405081e-01
min,-1.102923e+01
1%,-1.812765e+00
5%,-1.143114e+00
10%,-8.523545e-01
25%,-4.150700e-01
50%,2.904613e-02
75%,4.766843e-01


In [57]:
def last_threshold_signal(
    values,
    lower=LOWER_THRESHOLD,
    upper=UPPER_THRESHOLD,
    atr_column=ATR_DISTANCE_COLUMN,
):
    """Mark only the final row of each consecutive in-range sequence."""
    in_range = values[atr_column].between(lower, upper, inclusive="neither")
    return in_range & ~in_range.shift(-1, fill_value=False)


def day_probability(symbols):
    day_counts = {outcome: 0 for outcome in OUTCOMES}
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc="Classifying final threshold days",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        # Keep all chronological rows while identifying consecutive sequences.
        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
            & values[OPEN_COLUMN].notna()
            & values[CLOSE_COLUMN].notna()
        )
        qualifying = values.loc[signal]

        day_counts["green"] += (
            qualifying[CLOSE_COLUMN] > qualifying[OPEN_COLUMN]
        ).sum()
        day_counts["red"] += (
            qualifying[CLOSE_COLUMN] < qualifying[OPEN_COLUMN]
        ).sum()
        day_counts["unchanged"] += (
            qualifying[CLOSE_COLUMN] == qualifying[OPEN_COLUMN]
        ).sum()

    total_days = sum(day_counts.values())
    if total_days == 0:
        raise ValueError(
            f"No valid final rows have {LOWER_THRESHOLD} < "
            f"{ATR_DISTANCE_COLUMN} < {UPPER_THRESHOLD}."
        )

    result = pd.DataFrame(
        {
            "count": day_counts,
            "probability": {
                outcome: count / total_days
                for outcome, count in day_counts.items()
            },
        }
    )
    result.index.name = "outcome"
    return result


day_probability(symbols)

Classifying final threshold days:   0%|          | 0/3022 [00:00<?, ?symbol/s]

Classifying final threshold days: 100%|██████████| 3022/3022 [00:58<00:00, 51.60symbol/s]


,count,probability
outcome,,
green,45336,0.871092
red,6269,0.120453
unchanged,440,0.008454


Forward probabilities from last signal day.

In [58]:
# For each final threshold signal, compare its open with the close n rows later.
def forward_probability(symbols, n=N):
    horizons = range(n)
    forward_counts = {
        horizon: {outcome: 0 for outcome in OUTCOMES}
        for horizon in horizons
    }
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc="Calculating final-signal forward probabilities",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
            & values[OPEN_COLUMN].notna()
        )
        signal_open = values[OPEN_COLUMN]

        for horizon in horizons:
            future_close = values[CLOSE_COLUMN].shift(-horizon)
            valid = signal & future_close.notna()

            forward_counts[horizon]["green"] += (
                future_close[valid] > signal_open[valid]
            ).sum()
            forward_counts[horizon]["red"] += (
                future_close[valid] < signal_open[valid]
            ).sum()
            forward_counts[horizon]["unchanged"] += (
                future_close[valid] == signal_open[valid]
            ).sum()

    result = pd.DataFrame.from_dict(forward_counts, orient="index")
    result.index.name = "n"
    result["observations"] = result[list(OUTCOMES)].sum(axis=1)

    for outcome in OUTCOMES:
        result[f"{outcome}_probability"] = (
            result[outcome] / result["observations"].replace(0, np.nan)
        )

    return result[
        [
            "observations",
            "green",
            "green_probability",
            "red",
            "red_probability",
            "unchanged",
            "unchanged_probability",
        ]
    ]


forward_probability(symbols)

Calculating final-signal forward probabilities: 100%|██████████| 3022/3022 [02:05<00:00, 23.99symbol/s]


,observations,green,green_probability,red,red_probability,unchanged,unchanged_probability
n,,,,,,,
0,52045,45336,0.871092,6269,0.120453,440,0.008454
1,52042,36353,0.698532,15206,0.292187,483,0.009281
2,52033,33588,0.645513,18038,0.346665,407,0.007822
3,52018,32290,0.620747,19362,0.372217,366,0.007036
4,52010,31506,0.605768,20167,0.387752,337,0.006480
5,52001,31176,0.599527,20545,0.395089,280,0.005385
6,51997,30688,0.590188,20986,0.403600,323,0.006212
7,51991,30797,0.592353,20932,0.402608,262,0.005039
8,51985,30456,0.585861,21284,0.409426,245,0.004713


In [59]:
def regime_day_probability(symbols, regime_column=REGIME_COLUMN):
    """Calculate same-day probabilities for final signals by MA-short regime."""
    counts = {
        regime: {outcome: 0 for outcome in OUTCOMES}
        for regime in REGIME_LABELS
    }
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
        regime_column,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc="Classifying final signals by MA regime",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
            & values[OPEN_COLUMN].notna()
            & values[CLOSE_COLUMN].notna()
            & values[regime_column].isin(REGIME_LABELS)
        )

        for regime in REGIME_LABELS:
            regime_rows = values.loc[signal & (values[regime_column] == regime)]
            counts[regime]["green"] += (
                regime_rows[CLOSE_COLUMN] > regime_rows[OPEN_COLUMN]
            ).sum()
            counts[regime]["red"] += (
                regime_rows[CLOSE_COLUMN] < regime_rows[OPEN_COLUMN]
            ).sum()
            counts[regime]["unchanged"] += (
                regime_rows[CLOSE_COLUMN] == regime_rows[OPEN_COLUMN]
            ).sum()

    output_columns = {}
    for regime, label in REGIME_LABELS.items():
        total = sum(counts[regime].values())
        if total == 0:
            raise ValueError(
                f"No valid final {label} signals have {LOWER_THRESHOLD} < "
                f"{ATR_DISTANCE_COLUMN} < {UPPER_THRESHOLD}."
            )

        output_columns[(label, "count")] = pd.Series(counts[regime])
        output_columns[(label, "probability")] = pd.Series(
            {
                outcome: counts[regime][outcome] / total
                for outcome in OUTCOMES
            }
        )

    result = pd.DataFrame(output_columns).reindex(OUTCOMES)
    result.columns = pd.MultiIndex.from_tuples(
        result.columns,
        names=["regime", "metric"],
    )
    result.index.name = "outcome"
    return result


regime_day_probability(symbols)

Classifying final signals by MA regime: 100%|██████████| 3022/3022 [01:08<00:00, 44.32symbol/s]


regime    fanned             crossed            
metric     count probability   count probability
outcome                                         
green      12126    0.912896   14190    0.853072
red         1058    0.079651    2270    0.136467
unchanged     99    0.007453     174    0.010461

In [60]:
def regime_forward_probability(
    symbols,
    regime_column=REGIME_COLUMN,
    n=N,
):
    """Calculate forward probabilities for final signals by signal-date regime."""
    horizons = range(n)
    counts = {
        regime: {
            horizon: {outcome: 0 for outcome in OUTCOMES}
            for horizon in horizons
        }
        for regime in REGIME_LABELS
    }
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
        regime_column,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc="Calculating final-signal probabilities by MA regime",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        # Keep all chronological rows so both sequence endings and shift(-n)
        # retain their intended trading-row meaning.
        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        signal_open = values[OPEN_COLUMN]
        base_signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
            & values[OPEN_COLUMN].notna()
            & values[regime_column].isin(REGIME_LABELS)
        )

        for horizon in horizons:
            future_close = values[CLOSE_COLUMN].shift(-horizon)

            for regime in REGIME_LABELS:
                valid = (
                    base_signal
                    & (values[regime_column] == regime)
                    & future_close.notna()
                )
                counts[regime][horizon]["green"] += (
                    future_close[valid] > signal_open[valid]
                ).sum()
                counts[regime][horizon]["red"] += (
                    future_close[valid] < signal_open[valid]
                ).sum()
                counts[regime][horizon]["unchanged"] += (
                    future_close[valid] == signal_open[valid]
                ).sum()

    output_columns = {}
    for regime, label in REGIME_LABELS.items():
        observations = pd.Series(
            {
                horizon: sum(counts[regime][horizon].values())
                for horizon in horizons
            }
        )
        output_columns[(label, "observations")] = observations

        for outcome in OUTCOMES:
            outcome_counts = pd.Series(
                {
                    horizon: counts[regime][horizon][outcome]
                    for horizon in horizons
                }
            )
            output_columns[(label, f"{outcome}_count")] = outcome_counts
            output_columns[(label, f"{outcome}_probability")] = (
                outcome_counts / observations.replace(0, np.nan)
            )

    result = pd.DataFrame(output_columns)
    result.columns = pd.MultiIndex.from_tuples(
        result.columns,
        names=["regime", "metric"],
    )
    result.index.name = "n"
    return result


regime_forward_probability(symbols)

Calculating final-signal probabilities by MA regime: 100%|██████████| 3022/3022 [03:47<00:00, 13.27symbol/s]


regime       fanned                                                          \
metric observations green_count green_probability red_count red_probability   
n                                                                             
0             13283       12126          0.912896      1058        0.079651   
1             13283        9758          0.734623      3399        0.255891   
2             13282        8994          0.677157      4168        0.313808   
3             13278        8592          0.647085      4588        0.345534   
4             13277        8248          0.621225      4944        0.372373   
5             13276        8245          0.621045      4940        0.372100   
6             13275        8158          0.614539      5017        0.377928   
7             13273        8256          0.622015      4949        0.372862   
8             13273        8171          0.615611      5029        0.378889   
9             13271        8125          0.612237      5084        0.383091   
10            13271        8108          0.610956      5091        0.383618   

regime                                            crossed              \
metric unchanged_count unchanged_probability observations green_count   
n                                                                       
0                   99              0.007453        16634       14190   
1                  126              0.009486        16633       11072   
2                  120              0.009035        16630       10082   
3                   98              0.007381        16625        9621   
4                   85              0.006402        16623        9540   
5                   91              0.006854        16618        9520   
6                  100              0.007533        16617        9396   
7                   68              0.005123        16616        9512   
8                   73              0.005500        16613        9408   
9                   62              0.004672        16610        9272   
10                  72              0.005425        16607        9142   

regime                                                              \
metric green_probability red_count red_probability unchanged_count   
n                                                                    
0               0.853072      2270        0.136467             174   
1               0.665665      5395        0.324355             166   
2               0.606254      6437        0.387072             111   
3               0.578707      6877        0.413654             127   
4               0.573904      6969        0.419238             114   
5               0.572873      7007        0.421651              91   
6               0.565445      7113        0.428056             108   
7               0.572460      7017        0.422304              87   
8               0.566303      7124        0.428821              81   
9               0.558218      7253        0.436665              85   
10              0.550491      7377        0.444210              88   

regime                        
metric unchanged_probability  
n                             
0                   0.010461  
1                   0.009980  
2                   0.006675  
3                   0.007639  
4                   0.006858  
5                   0.005476  
6                   0.006499  
7                   0.005236  
8                   0.004876  
9                   0.005117  
10                  0.005299

Probabailities from same day values.

In [61]:
def forward_daily_candle_probability(symbols, n=N):
    """Calculate each future day's candle color after a final threshold signal."""
    horizons = range(n)
    counts = {
        horizon: {outcome: 0 for outcome in OUTCOMES}
        for horizon in horizons
    }
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc="Calculating forward daily candle probabilities",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
        )

        for horizon in horizons:
            future_open = values[OPEN_COLUMN].shift(-horizon)
            future_close = values[CLOSE_COLUMN].shift(-horizon)
            valid = signal & future_open.notna() & future_close.notna()

            counts[horizon]["green"] += (
                future_close[valid] > future_open[valid]
            ).sum()
            counts[horizon]["red"] += (
                future_close[valid] < future_open[valid]
            ).sum()
            counts[horizon]["unchanged"] += (
                future_close[valid] == future_open[valid]
            ).sum()

    result = pd.DataFrame.from_dict(counts, orient="index")
    result.index.name = "n"
    result["observations"] = result[list(OUTCOMES)].sum(axis=1)

    for outcome in OUTCOMES:
        result[f"{outcome}_probability"] = (
            result[outcome] / result["observations"].replace(0, np.nan)
        )

    return result[
        [
            "observations",
            "green",
            "green_probability",
            "red",
            "red_probability",
            "unchanged",
            "unchanged_probability",
        ]
    ]


forward_daily_candle_probability(symbols)

Calculating forward daily candle probabilities: 100%|██████████| 3022/3022 [01:34<00:00, 31.84symbol/s]


,observations,green,green_probability,red,red_probability,unchanged,unchanged_probability
n,,,,,,,
0,52045,45336,0.871092,6269,0.120453,440,0.008454
1,52042,15353,0.295012,35801,0.687925,888,0.017063
2,52033,23682,0.455134,27419,0.526954,932,0.017912
3,52018,23995,0.461283,27034,0.519705,989,0.019013
4,52010,24179,0.464891,26853,0.516305,978,0.018804
5,52001,24629,0.473626,26382,0.507336,990,0.019038
6,51997,24489,0.470969,26414,0.507991,1094,0.021040
7,51991,24982,0.480506,25994,0.499971,1015,0.019523
8,51985,24294,0.467327,26611,0.511898,1080,0.020775


In [62]:
def regime_forward_daily_candle_probability(
    symbols,
    regime_column=REGIME_COLUMN,
    n=N,
):
    """Calculate each future day's candle color by signal-date MA regime."""
    horizons = range(n)
    counts = {
        regime: {
            horizon: {outcome: 0 for outcome in OUTCOMES}
            for horizon in horizons
        }
        for regime in REGIME_LABELS
    }
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
        regime_column,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc="Calculating daily candle probabilities by MA regime",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        base_signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
            & values[regime_column].isin(REGIME_LABELS)
        )

        for horizon in horizons:
            future_open = values[OPEN_COLUMN].shift(-horizon)
            future_close = values[CLOSE_COLUMN].shift(-horizon)

            for regime in REGIME_LABELS:
                valid = (
                    base_signal
                    & (values[regime_column] == regime)
                    & future_open.notna()
                    & future_close.notna()
                )
                counts[regime][horizon]["green"] += (
                    future_close[valid] > future_open[valid]
                ).sum()
                counts[regime][horizon]["red"] += (
                    future_close[valid] < future_open[valid]
                ).sum()
                counts[regime][horizon]["unchanged"] += (
                    future_close[valid] == future_open[valid]
                ).sum()

    output_columns = {}
    for regime, label in REGIME_LABELS.items():
        observations = pd.Series(
            {
                horizon: sum(counts[regime][horizon].values())
                for horizon in horizons
            }
        )
        output_columns[(label, "observations")] = observations

        for outcome in OUTCOMES:
            outcome_counts = pd.Series(
                {
                    horizon: counts[regime][horizon][outcome]
                    for horizon in horizons
                }
            )
            output_columns[(label, f"{outcome}_count")] = outcome_counts
            output_columns[(label, f"{outcome}_probability")] = (
                outcome_counts / observations.replace(0, np.nan)
            )

    result = pd.DataFrame(output_columns)
    result.columns = pd.MultiIndex.from_tuples(
        result.columns,
        names=["regime", "metric"],
    )
    result.index.name = "n"
    return result


regime_forward_daily_candle_probability(symbols)

Calculating daily candle probabilities by MA regime:   0%|          | 0/3022 [00:00<?, ?symbol/s]

Calculating daily candle probabilities by MA regime: 100%|██████████| 3022/3022 [03:20<00:00, 15.05symbol/s]


regime       fanned                                                          \
metric observations green_count green_probability red_count red_probability   
n                                                                             
0             13283       12126          0.912896      1058        0.079651   
1             13283        3825          0.287962      9210        0.693367   
2             13282        6061          0.456332      6947        0.523039   
3             13278        6167          0.464452      6834        0.514686   
4             13277        6114          0.460496      6875        0.517813   
5             13276        6374          0.480114      6594        0.496686   
6             13275        6337          0.477363      6616        0.498380   
7             13273        6500          0.489716      6478        0.488058   
8             13273        5970          0.449785      7004        0.527688   
9             13271        6261          0.471781      6695        0.504483   
10            13271        6249          0.470876      6707        0.505388   

regime                                            crossed              \
metric unchanged_count unchanged_probability observations green_count   
n                                                                       
0                   99              0.007453        16634       14190   
1                  248              0.018670        16633        4891   
2                  274              0.020629        16630        7439   
3                  277              0.020862        16625        7667   
4                  288              0.021692        16623        7649   
5                  308              0.023200        16618        7814   
6                  322              0.024256        16617        7953   
7                  295              0.022226        16616        7956   
8                  299              0.022527        16613        7919   
9                  315              0.023736        16610        7493   
10                 315              0.023736        16607        7675   

regime                                                              \
metric green_probability red_count red_probability unchanged_count   
n                                                                    
0               0.853072      2270        0.136467             174   
1               0.294054     11440        0.687789             302   
2               0.447324      8891        0.534636             300   
3               0.461173      8596        0.517053             362   
4               0.460146      8655        0.520664             319   
5               0.470213      8476        0.510049             328   
6               0.478606      8292        0.499007             372   
7               0.478816      8300        0.499519             360   
8               0.476675      8318        0.500692             376   
9               0.451114      8741        0.526249             376   
10              0.462155      8561        0.515506             371   

regime                        
metric unchanged_probability  
n                             
0                   0.010461  
1                   0.018157  
2                   0.018040  
3                   0.021774  
4                   0.019190  
5                   0.019738  
6                   0.022387  
7                   0.021666  
8                   0.022633  
9                   0.022637  
10                  0.022340

In [63]:
def _new_expectancy_bucket():
    return {
        "observations": 0,
        "wins": 0,
        "losses": 0,
        "breakevens": 0,
        "winning_return_sum": 0.0,
        "losing_return_sum": 0.0,
    }


def _update_expectancy_bucket(bucket, returns):
    wins = returns[returns > 0]
    losses = returns[returns < 0]

    bucket["observations"] += len(returns)
    bucket["wins"] += len(wins)
    bucket["losses"] += len(losses)
    bucket["breakevens"] += (returns == 0).sum()
    bucket["winning_return_sum"] += wins.sum()
    bucket["losing_return_sum"] += losses.sum()


def _expectancy_results(stats):
    rows = {}
    for horizon, bucket in stats.items():
        observations = bucket["observations"]
        wins = bucket["wins"]
        losses = bucket["losses"]
        breakevens = bucket["breakevens"]

        win_probability = wins / observations if observations else np.nan
        loss_probability = losses / observations if observations else np.nan
        breakeven_probability = (
            breakevens / observations if observations else np.nan
        )
        average_win = (
            bucket["winning_return_sum"] / wins if wins else 0.0
        )
        average_loss = (
            -bucket["losing_return_sum"] / losses if losses else 0.0
        )
        expectancy = (
            win_probability * average_win
            - loss_probability * average_loss
        )

        rows[horizon] = {
            "observations": observations,
            "wins": wins,
            "win_probability": win_probability,
            "average_win_pct": average_win * 100,
            "losses": losses,
            "loss_probability": loss_probability,
            "average_loss_pct": average_loss * 100,
            "breakevens": breakevens,
            "breakeven_probability": breakeven_probability,
            "expectancy_pct": expectancy * 100,
        }

    result = pd.DataFrame.from_dict(rows, orient="index")
    result.index.name = "n"
    return result


def forward_daily_expectancy(symbols, trade_direction, n=N):
    """Calculate same-day open-to-close expectancy at each post-signal horizon."""
    direction = trade_direction.lower()
    if direction not in {"long", "short"}:
        raise ValueError("trade_direction must be 'long' or 'short'.")

    horizons = range(n)
    stats = {horizon: _new_expectancy_bucket() for horizon in horizons}
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc=f"Calculating {direction} daily expectancy",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
        )

        for horizon in horizons:
            future_open = values[OPEN_COLUMN].shift(-horizon)
            future_close = values[CLOSE_COLUMN].shift(-horizon)
            valid = (
                signal
                & future_open.notna()
                & future_close.notna()
                & (future_open != 0)
            )
            returns = (
                future_close[valid] - future_open[valid]
            ) / future_open[valid]
            if direction == "short":
                returns = -returns

            _update_expectancy_bucket(stats[horizon], returns)

    return _expectancy_results(stats)


forward_daily_expectancy(symbols, trade_direction=trade_direction)

Calculating short daily expectancy:   0%|          | 0/3022 [00:00<?, ?symbol/s]

Calculating short daily expectancy: 100%|██████████| 3022/3022 [02:18<00:00, 21.75symbol/s]


,observations,wins,win_probability,average_win_pct,losses,loss_probability,average_loss_pct,breakevens,breakeven_probability,expectancy_pct
n,,,,,,,,,,
0,52045,6269,0.120453,3.204580,45336,0.871092,7.039480,440,0.008454,-5.746034
1,52042,35801,0.687925,4.097556,15353,0.295012,4.465058,888,0.017063,1.501567
2,52033,27419,0.526954,3.773197,23682,0.455134,3.994820,932,0.017912,0.170122
3,52018,27034,0.519705,3.636008,23995,0.461283,3.904985,989,0.019013,0.088349
4,52010,26853,0.516305,3.581214,24179,0.464891,3.839370,978,0.018804,0.064107
5,52001,26382,0.507336,3.524256,24629,0.473626,3.726649,990,0.019038,0.022947
6,51997,26414,0.507991,3.480347,24489,0.470969,3.794562,1094,0.021040,-0.019139
7,51991,25994,0.499971,3.381467,24982,0.480506,3.751819,1015,0.019523,-0.112137
8,51985,26611,0.511898,3.403183,24294,0.467327,3.616096,1080,0.020775,0.052182


In [64]:
def regime_forward_daily_expectancy(
    symbols,
    trade_direction,
    regime_column=REGIME_COLUMN,
    n=N,
):
    """Calculate daily expectancy by the regime present on each signal date."""
    direction = trade_direction.lower()
    if direction not in {"long", "short"}:
        raise ValueError("trade_direction must be 'long' or 'short'.")

    horizons = range(n)
    stats = {
        regime: {
            horizon: _new_expectancy_bucket()
            for horizon in horizons
        }
        for regime in REGIME_LABELS
    }
    required = [
        OPEN_COLUMN,
        CLOSE_COLUMN,
        ATR_DISTANCE_COLUMN,
        RELATIVE_ATR_COLUMN,
        regime_column,
    ]

    for symbol_data in tqdm(
        symbols.values(),
        total=len(symbols),
        desc=f"Calculating {direction} daily expectancy by MA regime",
        unit="symbol",
    ):
        df = symbol_data.df
        if not all(column in df.columns for column in required):
            continue

        values = df.sort_index()[required].apply(pd.to_numeric, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        base_signal = (
            last_threshold_signal(values)
            & (values[RELATIVE_ATR_COLUMN] > RELATIVE_ATR_THRESHOLD)
            & values[regime_column].isin(REGIME_LABELS)
        )

        for horizon in horizons:
            future_open = values[OPEN_COLUMN].shift(-horizon)
            future_close = values[CLOSE_COLUMN].shift(-horizon)

            for regime in REGIME_LABELS:
                valid = (
                    base_signal
                    & (values[regime_column] == regime)
                    & future_open.notna()
                    & future_close.notna()
                    & (future_open != 0)
                )
                returns = (
                    future_close[valid] - future_open[valid]
                ) / future_open[valid]
                if direction == "short":
                    returns = -returns

                _update_expectancy_bucket(stats[regime][horizon], returns)

    regime_results = {
        label: _expectancy_results(stats[regime])
        for regime, label in REGIME_LABELS.items()
    }
    result = pd.concat(regime_results, axis=1)
    result.columns.names = ["regime", "metric"]
    return result


regime_forward_daily_expectancy(
    symbols,
    trade_direction=trade_direction,
)

Calculating short daily expectancy by MA regime: 100%|██████████| 3022/3022 [04:30<00:00, 11.17symbol/s]


regime       fanned                                               \
metric observations  wins win_probability average_win_pct losses   
n                                                                  
0             13283  1058        0.079651        3.050600  12126   
1             13283  9210        0.693367        4.146641   3825   
2             13282  6947        0.523039        3.775048   6061   
3             13278  6834        0.514686        3.509395   6167   
4             13277  6875        0.517813        3.515708   6114   
5             13276  6594        0.496686        3.468443   6374   
6             13275  6616        0.498380        3.471346   6337   
7             13273  6478        0.488058        3.282676   6500   
8             13273  7004        0.527688        3.320376   5970   
9             13271  6695        0.504483        3.421496   6261   
10            13271  6707        0.505388        3.412405   6249   

regime                                                                     \
metric loss_probability average_loss_pct breakevens breakeven_probability   
n                                                                           
0              0.912896         7.827632         99              0.007453   
1              0.287962         4.005575        248              0.018670   
2              0.456332         3.888506        274              0.020629   
3              0.464452         3.799760        277              0.020862   
4              0.460496         3.764081        288              0.021692   
5              0.480114         3.503188        308              0.023200   
6              0.477363         3.842145        322              0.024256   
7              0.489716         3.765641        295              0.022226   
8              0.449785         3.448139        299              0.022527   
9              0.471781         3.535107        315              0.023736   
10             0.470876         3.639275        315              0.023736   

regime                     crossed                                         \
metric expectancy_pct observations   wins win_probability average_win_pct   
n                                                                           
0           -6.902833        16634   2270        0.136467        3.291969   
1            1.721693        16633  11440        0.687789        4.209430   
2            0.200047        16630   8891        0.534636        3.822716   
3            0.041428        16625   8596        0.517053        3.693510   
4            0.087136        16623   8655        0.520664        3.668700   
5            0.040795        16618   8476        0.510049        3.560716   
6           -0.104049        16617   8292        0.499007        3.491827   
7           -0.241957        16616   8300        0.499519        3.386011   
8            0.201200        16613   8318        0.500692        3.427697   
9            0.058293        16610   8741        0.526249        3.557765   
10           0.010939        16607   8561        0.515506        3.501640   

regime                                                      \
metric losses loss_probability average_loss_pct breakevens   
n                                                            
0       14190         0.853072         6.685290        174   
1        4891         0.294054         4.558798        302   
2        7439         0.447324         4.030512        300   
3        7667         0.461173         3.905194        362   
4        7649         0.460146         3.964536        319   
5        7814         0.470213         3.857250        328   
6        7953         0.478606         3.864911        372   
7        7956         0.478816         3.717512        360   
8        7919         0.476675         3.692495        376   
9        7493         0.451114         3.620028        376   
10       7675         0.462155         3.624420        371   

regime              